← [10 - Cryptarithmes](Z3-10-Cryptarithmetic-CSharp.ipynb) | [README Z3-API](README.md) | [13b - UNSAT cores : le MUS >>](Z3-13b-UnsatCores-MUS-CSharp.ipynb)
# 12. Arithmetique reelle : raisonner sur les irrationnels exacts
Jusqu'ici nous avons raisonne sur des **entiers** (coloration de graphe, cryptarithmes, ordonnancement). Z3 sait aussi raisonner sur les **reels** via la théorie `Real` (SMT-LIB `Reals`). C'est une capacite profondement différente : les solutions peuvent etre **rationnelles exactes** (pas d'arrondi flottant), des **irrationnels algebriques** (racine carree de 2 representee comme racine d'un polynome, pas comme 1.4142...), et l'absence de solution peut etre **prouvee** (`unsat`) sur le corps des reels tout entier.
Ce notebook explore les trois postures : arithmetique lineaire (solution rationnelle exacte), non-lineaire (irrationnel algebrique), et preuve d'impossibilite sur R. Aucun `float`, aucun `sqrt()` approche : tout est symbolique.
> Jumeau C# du notebook Python [`Z3-12-Real-Arithmetic-Python.ipynb`](Z3-12-Real-Arithmetic-Python.ipynb). Il utilise le binding natif Microsoft.Z3 ; la section B7 Rational du notebook C# [`16_RealArithmetic.ipynb`](../Z3-Linq2Z3/16_RealArithmetic.ipynb), specifique au DSL Z3.Linq, n'est pas portee.

In [1]:
#r "nuget: Microsoft.Z3, 4.12.2"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);

// Contexte Z3 partage par toutes les cellules (meme moteur C++ que z3-solver Python)
var ctx = new Context();
Console.WriteLine($"Moteur : Microsoft.Z3 {Microsoft.Z3.Version.FullVersion}");
Console.WriteLine("Imports OK : Microsoft.Z3 (theorie Real)");

Installing Packages Microsoft.Z3

Moteur : Microsoft.Z3 Z3 4.12.2.0


Imports OK : Microsoft.Z3 (theorie Real)


### Lecture : la théorie se choisit à la déclaration, pas à l'import

Le message d'import nomme la théorie, mais `#r "nuget: Microsoft.Z3, 4.12.2"` n'en fixe aucune : il ne
charge que l'API. C'est la **sorte** des variables qui décide du fragment logique
interrogé. `ctx.MkRealConst("x")` place le solveur dans l'arithmétique des réels — un corps réel
clos, donc décidable ; `ctx.MkIntConst("x")` le mettrait dans l'arithmétique de Presburger ou de
Peano selon les opérations employées, avec des propriétés de décidabilité très
différentes ; `ctx.MkBitVecConst("x", 32)` ouvrirait la théorie des bit-vectors, où l'addition est
modulo `2^32`.

Cette remarque commande toute la lecture du notebook : les résultats qui suivent ne
dépendent pas du solveur mais de la théorie dans laquelle on l'a placé. Un énoncé faux
en arithmétique non bornée peut devenir vrai et décidable en arithmétique modulaire —
c'est précisément le sujet du notebook 14 consacré aux bit-vectors.

## 1. La théorie des reels (SMT-LIB `Real`)

En Microsoft.Z3, `ctx.MkRealConst("x")` créé une variable reelle (théorie des rationnels etant donne que Z3 represente les reels comme des rationnels ou des racines algebriques). Les opérations `ctx.MkAdd`, `ctx.MkMul`, comparaisons `ctx.MkGe` et egalites `ctx.MkEq` s'appliquent. Le solveur decide la satisfiabilite sur cette théorie via des procedure de decision (arithmetique reelle lineaire = simplex ; non-lineaire = méthodes de cylindres algebriques).

## 2. Arithmetique reelle lineaire : solution rationnelle exacte

Système lineaire `x + y = 1`, `x - y = 3`. Solution `x = 2`, `y = -1`. Ce sont des **rationnels exacts**, pas des flottants : Z3 ne fait jamais d'arrondi. Tout le raisonnement est symbolique.

In [2]:
// Systeme lineaire x+y=1, x-y=3 : solution rationnelle EXACTE
var x = ctx.MkRealConst("x");
var y = ctx.MkRealConst("y");
var s1 = ctx.MkSolver();
s1.Assert(ctx.MkEq(ctx.MkAdd(x, y), ctx.MkReal(1)));
s1.Assert(ctx.MkEq(ctx.MkSub(x, y), ctx.MkReal(3)));
var st1 = s1.Check();
Console.WriteLine($"Systeme x+y=1, x-y=3 : {st1}");
if (st1 == Status.SATISFIABLE)
{
    var m = s1.Model;
    Console.WriteLine($"  x = {m.Evaluate(x, true)}");
    Console.WriteLine($"  y = {m.Evaluate(y, true)}");
}
Console.WriteLine("-> Solution rationnelle EXACTE (pas d'arrondi flottant).");

Systeme x+y=1, x-y=3 : SATISFIABLE


  x = 2


  y = -1


-> Solution rationnelle EXACTE (pas d'arrondi flottant).


### Lecture : ce que « x = 2 » veut dire exactement

Le modèle rend `2` et `-1`, pas `2.0` et `-1.0`. L'écart n'est pas cosmétique : `Real`
désigne la théorie des corps réels clos, où Z3 représente les valeurs comme des
rationnels **exacts** — numérateur et dénominateur entiers — jamais comme des flottants
binaires. Là où une résolution numérique rendrait `2.0000000e+00` avec un résidu
d'arrondi, la réponse d'ici est exacte par construction : substituer `2` et `-1` dans
`x + y = 1` et `x - y = 3` produit des égalités **vraies**, pas des égalités à epsilon
près.

La conséquence pratique est plus forte qu'il n'y paraît : un `Check()` qui rend
`SATISFIABLE` sur un système linéaire certifie l'existence d'une solution exacte *et* en fournit une.
C'est ce qui rend Z3 utilisable quand les coefficients sont des rationnels non
représentables en binaire — précisément le cas où un solveur flottant dérive.

## 3. Non-lineaire : trouver un irrationnel exact (racine de 2)

Trouvons `xs` tel que `xs^2 = 2`, avec `xs > 0` (la racine positive). La solution n'est PAS le flottant `1.41421356...` : Z3 renvoie une **racine algebrique exacte** (un `root-obj` = k-ieme racine d'un polynome a coefficients rationnels), affichée ici en notation decimale avec un `?` final qui signale "representation approchee d'un nombre exact".

In [3]:
// Trouver xs tel que xs^2 = 2, xs > 0 (l'irrationnel sqrt(2))
var xs = ctx.MkRealConst("xs");
var s2 = ctx.MkSolver();
s2.Assert(ctx.MkEq(ctx.MkMul(xs, xs), ctx.MkReal(2)));
s2.Assert(ctx.MkGt(xs, ctx.MkReal(0)));
var st2 = s2.Check();
Console.WriteLine($"xs^2 = 2, xs > 0 : {st2}");
if (st2 == Status.SATISFIABLE)
{
    var m = s2.Model;
    var w = m.Evaluate(xs, true);
    Console.WriteLine($"  xs = {w}");
    Console.WriteLine("  -> Ce n'est PAS 1.4142... mais une RACINE ALGEBRIQUE exacte.");
    Console.WriteLine($"     (objet Microsoft.Z3 {w.GetType().Name} : {w} ; x^2 - 2 = 0)");
}

xs^2 = 2, xs > 0 : SATISFIABLE


  xs = (root-obj (+ (^ x 2) (- 2)) 2)


  -> Ce n'est PAS 1.4142... mais une RACINE ALGEBRIQUE exacte.


     (objet Microsoft.Z3 AlgebraicNum : (root-obj (+ (^ x 2) (- 2)) 2) ; x^2 - 2 = 0)


### Lecture du `?` : `1.4142135623?` n'est pas un flottant

C'est le point le plus facile à mal lire du notebook. Z3 n'a **pas** rendu le flottant
`1.4142135623730951` : il a rendu un **objet-racine** (*root-obj*), et le `?` final est
la marque de l'affichage. Le modèle porte donc la solution sous forme algébrique
symbolique — la racine positive du polynôme minimal `x^2 - 2 = 0` — et la forme
décimale n'est qu'un rendu approché, produit au moment de l'affichage.

L'écart se mesure : `1.4142135623 ** 2` vaut `1.999999999793256`, donc à côté de `2` de
plus de `2e-10` ; l'objet-racine, lui, vérifie `ctx.MkMul(xs, xs) == ctx.MkReal(2)` **exactement**. C'est
toute la différence entre *représenter* un irrationnel et l'*approcher*. Le degré du
polynôme minimal est l'information qui compte ici — degré 2 pour la racine carrée — et
c'est ce qui donne son intérêt à l'exercice 1 : la racine cubique est de degré 3, et la
théorie la traite par le même mécanisme, sans changement d'API.

> En C#, Microsoft.Z3 expose cette valeur comme un `AlgebraicNum` et l'affiche sous la forme `(root-obj (+ (^ x 2) (- 2)) 2)`, plutôt que sous la notation décimale Python `1.4142135623?`. Le verdict et la valeur algébrique restent identiques.

## 4. Prouver l'absence de racine reelle (UNSAT)

`xn^2 + 1 = 0` n'a pas de solution reelle (le discriminant est negatif, les racines sont imaginaires `+/-i`). Z3 le **prouve** : `UNSATISFIABLE`. C'est une preuve sur le corps des reels tout entier - il n'existe AUCUN reel satisfaisant cette equation. Aucune approche numérique (essayer des valeurs, echantillonner) ne peut etablir cette absence ; seule la decision algebrique le peut.

In [4]:
// Prouver que xn^2 + 1 = 0 n'a PAS de solution reelle
var xn = ctx.MkRealConst("xn");
var s3 = ctx.MkSolver();
s3.Assert(ctx.MkEq(ctx.MkAdd(ctx.MkMul(xn, xn), ctx.MkReal(1)), ctx.MkReal(0)));
var st3 = s3.Check();
Console.WriteLine($"xn^2 + 1 = 0 sur les reels : {st3}");
if (st3 == Status.UNSATISFIABLE)
    Console.WriteLine("  -> UNSATISFIABLE : Z3 PROUVE qu'il n'existe pas de racine reelle.");
else
    Console.WriteLine("  -> inattendu (devrait etre impossible sur R).");

xn^2 + 1 = 0 sur les reels : UNSATISFIABLE


  -> UNSATISFIABLE : Z3 PROUVE qu'il n'existe pas de racine reelle.


### Lecture : `unsat` est une preuve, pas un échec de recherche

`unsat` se lit souvent comme « le solveur n'a rien trouvé ». Ici c'est l'inverse : sur
la théorie des réels, `unsat` signifie que la **négation** de l'énoncé est
insatisfiable, donc que l'énoncé est valide pour *toutes* les valuations. Z3 ne dit pas
« je n'ai pas trouvé de `xn` tel que `xn^2 + 1 = 0` » ; il dit qu'un tel `xn` n'existe
pas dans `R`. La traduction algébrique est immédiate : `xn^2 = -1` n'a pas de solution
réelle puisque le carré d'un réel est positif ou nul — le `discriminant < 0` du trinôme
dit exactement la même chose, vue par les coefficients.

Cette bascule `sat` / `unsat` est ce qui fait de Z3 un **prouveur** et non seulement un
chercheur de solutions. Elle sert directement à l'exercice 2, où discriminer deux
trinômes revient à comparer un `sat` et un `unsat` sur la même famille de formules.

## 5. Application geometrique : l'inegalite triangulaire

Trois longueurs `a, b, c` forment un triangle ssi chacune est <= la somme des deux autres (`a + b >= c`, etc.). Pour `(3, 4, 5)` c'est SATISFIABLE (triangle rectangle canonique) ; pour `(1, 2, 5)` c'est UNSATISFIABLE (1 + 2 = 3 < 5, le plus long cote est trop long). La preuve est exacte, pas une verification numérique.

In [5]:
// Trois longueurs forment-elles un triangle ? (inegalite triangulaire)
var a = ctx.MkRealConst("a");
var b = ctx.MkRealConst("b");
var c = ctx.MkRealConst("c");

BoolExpr Triangle(RealExpr va, RealExpr vb, RealExpr vc) => ctx.MkAnd(
    ctx.MkGe(ctx.MkAdd(va, vb), vc),
    ctx.MkGe(ctx.MkAdd(va, vc), vb),
    ctx.MkGe(ctx.MkAdd(vb, vc), va));

// Triangle (3, 4, 5) ?
var sA = ctx.MkSolver();
sA.Assert(Triangle(ctx.MkReal(3), ctx.MkReal(4), ctx.MkReal(5)));
Console.WriteLine($"Triangle(3, 4, 5) : {sA.Check()}");

// Triangle (1, 2, 5) ?
var sB = ctx.MkSolver();
sB.Assert(Triangle(ctx.MkReal(1), ctx.MkReal(2), ctx.MkReal(5)));
var stB = sB.Check();
Console.WriteLine($"Triangle(1, 2, 5) : {stB}");
if (stB == Status.UNSATISFIABLE)
    Console.WriteLine("  -> UNSATISFIABLE : impossible (1 + 2 = 3 < 5, le plus long cote est trop long).");

Triangle(3, 4, 5) : SATISFIABLE


Triangle(1, 2, 5) : UNSATISFIABLE


  -> UNSATISFIABLE : impossible (1 + 2 = 3 < 5, le plus long cote est trop long).


### Lecture : le même encodage rend `sat` puis `unsat`

Les deux appels partagent exactement la même formule `triangle(...)` — seule change la
valuation. `(3, 4, 5)` satisfait les trois inégalités ; `(1, 2, 5)` échoue sur celle qui
compte, `1 + 2 = 3 < 5`. Le point pédagogique est que le prédicat n'a **pas** été
réécrit pour le second cas : c'est la même conjonction, et c'est le solveur qui découvre
laquelle des trois inégalités tombe.

Notez la frontière exacte de cet encodage : avec des inégalités **larges**, un triplet
`(1, 2, 3)` rendrait `sat` alors que le triangle est plat — le cas dégénéré est accepté.
Exclure les triangles dégénérés demande la forme stricte. C'est le genre d'ajustement où
le solveur continue de donner la bonne réponse sans qu'on ait à énumérer les cas
limites à la main.

## 6. Ce que ce notebook a demontre

Trois capacites distinctes de la théorie des reels :
1. **Solution rationnelle exacte** (linearite) - pas de flottant, pas d'arrondi.

2. **Irrationnel algebrique exact** (non-lineaire) - racine de 2 comme `root-obj` (en C#, un `AlgebraicNum` affiche `(root-obj (+ (^ x 2) (- 2)) 2)`), pas comme decimal tronque.

3. **Preuve d'absence sur R** (`UNSATISFIABLE`) - aucune racine reelle a `x^2 + 1 = 0`, preuve globale impossible par echantillonnage.



Ces trois postures sont inaccessibles au calcul numérique classique (float, `Math.Sqrt`). Elles illustrent pourquoi la théorie SMT des reels est un outil de raisonnement, pas juste de calcul.

## 7. Exercices

Les trois exercices ci-dessous vous font manipuler la théorie des reels sur des variantes : racine cubique (1), discriminant d'un trinome (2), optimisation sur les reels (3). Chaque stub est self-contained : le contexte Z3 (`ctx`) et les fabriques (`ctx.MkRealConst`, `ctx.MkSolver`, `ctx.MkOptimize`) sont definis dans les sections précédentes.

### Exercice 1 - Racine cubique de 2
Trouvez `c` tel que `c^3 = 2`, avec `c > 0`. La racine cubique de 2 est aussi un irrationnel algebrique, mais de **degré 3** (vs degré 2 pour la racine carree). Affichez le temoin et comparez la notation.

**Étapes** : `c = ctx.MkRealConst("c")`, ajouter `ctx.MkEq(ctx.MkMul(ctx.MkMul(c, c), c), ctx.MkReal(2))` et `ctx.MkGt(c, ctx.MkReal(0))`, checker, afficher le modèle.

In [6]:
// Exercice 1 : trouver c tel que c^3 = 2 (racine cubique de 2)
// TODO etudiant :
// Etape 1 : var cEx1 = ctx.MkRealConst("c");
// Etape 2 : var sEx1 = ctx.MkSolver();
//            sEx1.Assert(ctx.MkEq(ctx.MkMul(ctx.MkMul(cEx1, cEx1), cEx1), ctx.MkReal(2)));
//            sEx1.Assert(ctx.MkGt(cEx1, ctx.MkReal(0)));
// Etape 3 : sEx1.Check() + afficher le modele (degre 3 vs degre 2 pour sqrt(2))
string resultatEx1 = null;  // TODO etudiant : remplacer par la valeur du modele (root-obj)
Console.WriteLine("Exercice 1 a completer : trouvez la racine cubique exacte de 2");
Console.WriteLine($"Resultat : {(resultatEx1 ?? "(a completer)")}");

Exercice 1 a completer : trouvez la racine cubique exacte de 2


Resultat : (a completer)


### Exercice 2 - Discriminant et nombre de racines
Un trinome `q^2 + bq + c = 0` a des racines reelles ssi son **discriminant** `D = b^2 - 4c >= 0`.
- (a) Prouvez que `q^2 - 3q + 2 = 0` a une racine reelle (`SATISFIABLE` ; deux racines : 1 et 2).
- (b) Prouvez que `q^2 + q + 1 = 0` n'en a aucune (`UNSATISFIABLE` ; D = -3 < 0).

**Étapes** : pour chaque polynome, encoder `= 0` avec `ctx.MkEq(...)` et checker (`ctx.MkSolver().Check()`). Le verdict `SATISFIABLE`/`UNSATISFIABLE` confirme le signe du discriminant.

In [7]:
// Exercice 2 : discriminant et racines reelles
// TODO etudiant :
// (a) q^2 - 3q + 2 = 0 a une racine reelle (SATISFIABLE, deux racines 1 et 2)
// (b) q^2 + q + 1 = 0 n'en a aucune (UNSATISFIABLE, D = -3 < 0)
// Etape 1 : encoder chaque polynome = 0 (ctx.MkEq(ctx.MkAdd(...), ctx.MkReal(0)))
// Etape 2 : ctx.MkSolver().Check() -> SATISFIABLE puis UNSATISFIABLE
string resultatEx2 = null;  // TODO etudiant : remplacer par les deux verdicts
Console.WriteLine("Exercice 2 a completer : discriminez les trinomes (SATISFIABLE vs UNSATISFIABLE)");
Console.WriteLine($"Resultat : {(resultatEx2 ?? "(a completer)")}");

Exercice 2 a completer : discriminez les trinomes (SATISFIABLE vs UNSATISFIABLE)


Resultat : (a completer)


### Exercice 3 - Distance minimale (optimisation sur les reels)
Minimisez `px^2 + py^2` sous la contrainte `px + py = 4` (point de la droite le plus proche de l'origine). La solution exacte est `px = py = 2`, distance minimale `sqrt(8)`.

**Indice** : utilisez `ctx.MkOptimize()` avec `opt.MkMinimize(ctx.MkAdd(ctx.MkMul(px, px), ctx.MkMul(py, py)))`. **Note** : l'optimisation non-lineaire sur les reels est delicate (Z3 peut renvoyer un point sous-optimal, et la resolution peut ne pas aboutir rapidement) ; si le minimum trouve n'est pas 8, reflechissez a pourquoi (la théorie non-lineaire reelle est decidable mais l'optimisation globale reste dure).

In [8]:
// Exercice 3 : minimiser px^2 + py^2 sous px + py = 4 (point le plus proche de l'origine)
// TODO etudiant :
// Etape 1 : var px = ctx.MkRealConst("px"); var py = ctx.MkRealConst("py");
// Etape 2 : var optEx3 = ctx.MkOptimize(); optEx3.Assert(ctx.MkEq(ctx.MkAdd(px, py), ctx.MkReal(4)));
// Etape 3 : optEx3.MkMinimize(ctx.MkAdd(ctx.MkMul(px, px), ctx.MkMul(py, py)));
// Etape 4 : optEx3.Check() -> minimum attendu = 8 en px = py = 2 (distance sqrt(8))
double? minimumEx3 = null;  // TODO etudiant : remplacer par le minimum trouve
Console.WriteLine("Exercice 3 a completer : minimisez x^2 + y^2 sous x + y = 4");
Console.WriteLine($"Resultat : {(minimumEx3.HasValue ? minimumEx3.Value.ToString() : "(a completer)")}");

Exercice 3 a completer : minimisez x^2 + y^2 sous x + y = 4


Resultat : (a completer)


## Conclusion
La théorie des reels de Z3 transforme le solveur en outil de **raisonnement algebrique exact**. La ou le calcul numérique flottant (`float`, `Math.Sqrt`) introduit des erreurs d'arrondi et ne peut jamais prouver une absence de solution, la théorie SMT `Real` offre : solutions rationnelles exactes, irrationnels algebriques (racine de 2 comme `root-obj`, un `AlgebraicNum` en Microsoft.Z3), et preuves d'impossibilite sur R tout entier (`UNSATISFIABLE` pour `x^2 + 1 = 0`).

Cette serie Z3-API (notebooks 01 a 12) couvre maintenant l'eventail complet : entiers (01-06, 10, 11), satisfaction de contraintes (08 ordonnancement, 09 CSP logique, 11 graphes), tactiques (03), et ici l'**arithmetique reelle exacte** (12). Ce carnet est le jumeau C# du notebook Python [`Z3-12-Real-Arithmetic-Python.ipynb`](Z3-12-Real-Arithmetic-Python.ipynb) : les memes exemples sont portes entre Microsoft.Z3 et pyz3, via le binding natif **Microsoft.Z3** (NuGet, v4.12.2.0).

***

> Carnet issu de l'EPIC #1206.